# Loading and preparing data 

In [1]:
import scirpy as ir
import scanpy as sc
import anndata as ad

import pandas as pd
import numpy as np
import os
import warnings

In [2]:
ir.__version__

'0.12.0'

In [ ]:
warnings.filterwarnings('ignore')

In [ ]:
# rpy2 
os.environ['R_HOME'] = '/nobackup/peer/fdeckert/miniconda3/envs/r.4.1.0-FD20220623HSCT/lib/R/'

In [ ]:
sc.set_figure_params(figsize=(7.5, 7.5), dpi_save=1200, fontsize=18, frameon=False)

In [ ]:
os.chdir('/research/peer/fdeckert/FD20220623HSCT')

# Load scirpy DB

In [ ]:
vdjdb = ir.datasets.vdjdb()
iedb = ir.datasets.iedb()

# Import AnnData

In [111]:
adata = sc.read_h5ad('data/object/pp.h5ad')

In [112]:
# sample path of 10x TCR data 
adata.obs['sample_path_tcr'] = np.where(adata.obs['digestion_protocol']=='old', adata.obs['sample_path'].astype(str).replace('_transcriptome', '_TCR', regex=True), adata.obs['sample_path'].astype(str).replace('_transcriptome', '_abTCR_VDJ', regex=True))
adata.obs['sample_path_tcr'] = adata.obs['sample_path_tcr'].replace('_TCR', '_VDJ', regex=True)

In [ ]:
# Plotting 
import rpy2.robjects as robjects
color_load = robjects.r.source('plotting_global.R')
color = dict()
for i in range(len(color_load[0])):
    color[color_load[0].names[i]] = {key : color_load[0][i].rx2(key)[0] for key in color_load[0][i].names}
    
# Set color function 
def set_color(categories): 
    
    categories = [x for x in categories if x in list(adata.obs.columns)]

    for category in categories: 

        adata.obs[category] = pd.Series(adata.obs[category], dtype='category').cat.remove_unused_categories()
        
        keys = list(color[category].keys())
        keys = [x for x in keys if x in list(adata.obs[category])]        

        adata.obs[category] = adata.obs[category].cat.reorder_categories(keys)
        adata.uns[category+'_colors'] = np.array([color[category].get(key) for key in keys], dtype=object)

# Imort TCR 

In [ ]:
adata_tcr_dict = []
for sample_path_tcr in adata.obs['sample_path_tcr'].unique(): 
    
    print(sample_path_tcr)
    
    # Import TCR 10x data
    adata_tcr = ir.io.read_10x_vdj(sample_path_tcr+'/filtered_contig_annotations.csv')
    adata_cnt = adata[adata.obs['sample_path_tcr']==sample_path_tcr]
    
    # Get cell_id index from meta data and add to adata_tcr
    barcode_idx = '_'+adata_cnt.obs_names[1].split('_')[1]
    adata_tcr.obs_names = adata_tcr.obs_names+barcode_idx

    # Subset for obs names union 
    obs_names_union = set(adata_tcr.obs_names) & set(adata_cnt.obs_names)
    adata_tcr = adata_tcr[adata_tcr.obs_names.isin(obs_names_union)]
    adata_cnt = adata_cnt[adata_cnt.obs_names.isin(obs_names_union)]
    
    # Add obs data to adata tcr
    adata_cnt.obs = adata_cnt.obs[list(set(adata_cnt.obs.columns) - set(adata_tcr.obs.columns))]
    adata_tcr.obs = adata_cnt.obs.join(adata_tcr.obs)
    
    # print(adata.obsm['X_umap'])
    adata_tcr.obsm['X_umap'] = adata_cnt.obsm['X_umap']
    
    # Output to dict
    adata_tcr_dict.append(adata_tcr)

In [ ]:
adata_tcr = ad.concat(adata_tcr_dict)

# Set colors

In [ ]:
set_color(categories=list(color.keys()))

# TCR QC 

In [ ]:
ir.tl.chain_qc(adata_tcr)

In [ ]:
ax = ir.pl.group_abundance(adata_tcr, groupby='chain_pairing', target_col='patient_id')

In [ ]:
ax = ir.pl.group_abundance(adata_tcr, groupby='receptor_subtype', target_col='patient_id')

# Subset by chain pairing 

In [ ]:
adata_tcr = adata_tcr[adata_tcr.obs['chain_pairing'].isin(['single pair', 'extra VJ', 'extra VDJ', 'two full chains']), :].copy()

# Compute clono types

In [ ]:
# Dist based on AA similarity 
ir.pp.ir_dist(adata_tcr, metric='alignment', sequence='aa', cutoff=10)
ir.tl(adata_tcr, metric='alignment', sequence='aa', receptor_arms='all', dual_ir='any')

In [ ]:
ir.tl.clonotype_network(adata_tcr, min_cells=2, sequence='aa', metric='alignment')
ir.pl.clonotype_network(adata_tcr, color='hto_demux_class', show_labels=False, panel_size=(10, 10), base_size=20, edges=True)

# Clonotype analysis

In [ ]:
ir.tl.clonal_expansion(adata_tcr, target_col='cc_aa_alignment', clip_at=2)

In [ ]:
sc.pl.umap(adata_tcr, color=['cell_type_leiden', 'clonal_expansion', 'cc_aa_alignment_size'], frameon=False, ncols=3, wspace=0.5, size=50)

# Match VDJDB

In [ ]:
ir.pp.ir_dist(adata_tcr, vdjdb, metric='identity', sequence='aa')
ir.tl.ir_query(adata_tcr, vdjdb, metric='identity', sequence='aa', receptor_arms='any', dual_ir='any')
ir.tl.ir_query_annotate(adata_tcr, vdjdb, metric='identity', sequence='aa', include_ref_cols=['antigen.species', 'antigen.gene'], strategy='most-frequent',)
adata_tcr.obs.rename(columns = {'antigen.species':'species_vdjdb', 'antigen.gene':'antigen_vdjdb'}, inplace = True)

In [ ]:
sc.pl.umap(adata_tcr, color=['species_vdjdb'], frameon=False, ncols=2, wspace=0.5, size=50)

In [ ]:
sc.pl.umap(adata_tcr, color=['antigen_vdjdb'], frameon=False, ncols=2, wspace=0.5, size=50)

# Match IEDB

In [ ]:
ir.pp.ir_dist(adata_tcr, iedb, metric='identity', sequence='aa')
ir.tl.ir_query(adata_tcr, iedb, metric='identity', sequence='aa', receptor_arms='any', dual_ir='any')
ir.tl.ir_query_annotate(adata_tcr, iedb, metric='identity', sequence='aa', include_ref_cols=['Epitope Source Molecule', 'Epitope Source Organism'], strategy='most-frequent',)
adata_tcr.obs.rename(columns = {'Epitope Source Organism':'species_iedb', 'Epitope Source Molecule':'antigen_iedb'}, inplace = True)

# Save results 

In [132]:
adata_tcr.obs[['has_ir', 'cc_aa_alignment', 'cc_aa_alignment_size'] + list(set(adata_tcr.obs.columns) - set(adata.obs.columns))].to_csv('result/scirpy/scirpy.csv')